# Pytorch Tutorial

Pytorch is a python framework for machine learning

- GPU-accelerated computations
- automatic differentiation
- modules for neural networks

This tutorial will teach you the fundamentals of operating on pytorch tensors and networks. You have already seen some things in recitation 0 which we will quickly review, but most of this tutorial is on mostly new or more advanced stuff.

For additional tutorials, see http://pytorch.org/tutorials/

For pytorch installation:
```pip3 install torch torchvision torchaudio```

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error, explained_variance_score

## Tensors (review)

Tensors are the fundamental object for array data. The most common types you will use are `IntTensor` and `FloatTensor`.

In [2]:
# Create uninitialized tensor
x = torch.FloatTensor(2,3)
print(x)
# Initialize to zeros
# x.zero_()
# print(x)

tensor([[0., 0., 0.],
        [0., 0., 0.]])


In [2]:
# Create from numpy array (seed for repeatability)
np.random.seed(123)
np_array = np.random.random((2,3))
print(torch.FloatTensor(np_array))
print(torch.from_numpy(np_array))

tensor([[0.6965, 0.2861, 0.2269],
        [0.5513, 0.7195, 0.4231]])
tensor([[0.6965, 0.2861, 0.2269],
        [0.5513, 0.7195, 0.4231]], dtype=torch.float64)


In [4]:
# Create random tensor (seed for repeatability)
torch.manual_seed(123)
x=torch.randn(2,3)
print(x)
# export to numpy array
x_np = x.numpy()
print(x_np)

tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093]])
[[-0.11146712  0.12036294 -0.3696345 ]
 [-0.24041797 -1.1969243   0.20926936]]


In [3]:
# special tensors (see documentation)
print(torch.eye(3))
print(torch.ones(2,3))
print(torch.zeros(2,3))
print(torch.arange(0,3))

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])
tensor([[1., 1., 1.],
        [1., 1., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([0, 1, 2])


All tensors have a `size` and `type`

In [4]:
x=torch.FloatTensor(3,4)
print(x.size())
print(x.type())

torch.Size([3, 4])
torch.FloatTensor


## Math, Linear Algebra, and Indexing (review)

Pytorch math and linear algebra is similar to numpy. Operators are overridden so you can use standard math operators (`+`,`-`, etc.) and expect a tensor as a result. See pytorch documentation for a complete list of available functions.

In [5]:
x = torch.arange(0,5)
print(x)
print('------------------')
print(torch.sum(x))
print(torch.sum(torch.exp(x)))
print(torch.mean(x.float()))

tensor([0, 1, 2, 3, 4])
------------------
tensor(10)
tensor(85.7910)
tensor(2.)


Pytorch indexing is similar to numpy indexing. See pytorch documentation for details.

In [6]:
x = torch.rand(3,2)
print(x)
print(x[1,:])

tensor([[0.3261, 0.8632],
        [0.9205, 0.3884],
        [0.6679, 0.6583]])
tensor([0.9205, 0.3884])


## Differentiation

Tensors provide automatic differentiation for back propagation.

As you might know, previous versions of Pytorch used Variables, which were wrappers around tensors for differentiation. Starting with pytorch 0.4.0, this wrapping is done internally in the Tensor class and you can, and should, differentiate Tensors directly. However, it is possible that you walk on references to Variables, e.g. in your error messages.

What you need to remember :

- Tensors you are differentiating with respect to must have `requires_grad=True`
- Call `.backward()` on scalar variables you are differentiating
- To differentiate a vector, sum it first

In [7]:
# Create differentiable tensor
x = torch.tensor(np.arange(0,4,dtype=np.float32), requires_grad=True)
# Calculate y=sum(x**2)
y = x**2
# Calculate gradient (dy/dx=2x)
y.sum().backward()
# Print values
print(x)
print(y)
print(x.grad)

tensor([0., 1., 2., 3.], requires_grad=True)
tensor([0., 1., 4., 9.], grad_fn=<PowBackward0>)
tensor([0., 2., 4., 6.])


Differentiation accumulates gradients. This is sometimes what you want and sometimes not. **Make sure to zero gradients between batches if performing gradient descent or you will get strange results!**

In [11]:
# Create a variable
x=torch.tensor(np.arange(0,4,dtype=np.float32), requires_grad=True)
# Differentiate
torch.sum(x**2).backward()
print(x.grad)
# Differentiate again (accumulates gradient)
torch.sum(x**2).backward()
print(x.grad)
# Zero gradient before differentiating
x.grad.data.zero_()
torch.sum(x**2).backward()
print(x.grad)

tensor([0., 2., 4., 6.])
tensor([ 0.,  4.,  8., 12.])
tensor([0., 2., 4., 6.])


Note that a Tensor with gradient cannot be exported to numpy directly :

In [12]:
x=torch.tensor(np.arange(0,4,dtype=np.float32), requires_grad=True)
x.numpy() # raises an exception

RuntimeError: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

The reason is that pytorch remembers the graph of all computations to perform differenciation. To be integrated to this graph the raw data is wrapped internally to the Tensor class (like what was formerly a Variable). You can detach the tensor from the graph using the **.detach()** method, which returns a tensor with the same data but requires_grad set to False.

In [8]:
x=torch.tensor(np.arange(0,4,dtype=np.float32), requires_grad=True)
y=x**2
z=y**2
z.detach().numpy()

array([ 0.,  1., 16., 81.], dtype=float32)

Another reason to use this method is that updating the graph can use a lot of memory. If you are in a context where you have a differentiable tensor that you don't need to differentiate, think of detaching it from the graph.

## Neural Network Modules - A Case Study of COVID prediction

Pytorch provides a framework for developing neural network modules. They take care of many things, the main one being wrapping and tracking a list of parameters for you.
In this case, data is included in a .csv file
The last element of a row is the label

In [ ]:
# data = pd.read_csv('./covid.train.csv')
# data = data.drop(columns=['id'])

In [ ]:
# data

In [ ]:
# from sklearn.model_selection import train_test_split
# train_df, test_df = train_test_split(data, test_size=0.15, random_state=42)

In [ ]:
# train_df.to_csv('covid.train.csv', index=False)
# test_df.to_csv('covid.test.csv', index=False)

In [2]:
train_data = pd.read_csv('./covid.train.csv').values
x_train, y_train = train_data[:,:-1], train_data[:,-1]
test_data = pd.read_csv('./covid.test.csv').values
x_test, y_test = test_data[:,:-1], test_data[:,-1]

In [3]:
sample_data = pd.read_csv('./covid.train.csv')
sample_data.head(10)

,AL,AK,AZ,AR,CA,CO,CT,FL,GA,ID,...,work_outside_home.4,shop.4,restaurant.4,spent_time.4,large_event.4,public_transit.4,anxious.4,depressed.4,worried_finances.4,tested_positive.4
0,0,0,1,0,0,0,0,0,0,0,...,29.626924,67.239624,37.772328,45.070882,20.053098,4.807546,9.688968,8.408900,33.946934,6.573774
1,0,0,0,0,0,0,0,0,0,0,...,38.074184,69.580523,39.023356,49.923518,24.234582,3.658905,11.460076,10.679890,33.456428,8.244562
2,0,0,0,0,0,0,0,0,0,0,...,32.402426,66.050550,34.444846,46.511627,19.748107,5.577020,10.349283,8.739371,34.576891,6.402434
3,0,0,0,0,0,0,0,0,0,0,...,33.368687,63.708127,34.080572,36.575378,17.316802,5.831276,13.216630,11.270658,37.622073,18.138676
4,0,0,0,0,0,0,0,0,0,0,...,38.982138,66.217163,36.153701,46.783154,21.737841,2.942482,13.246759,11.006826,32.627393,16.323035
5,0,0,0,0,0,0,0,0,0,0,...,28.356106,66.669563,29.426851,41.302956,13.959906,5.394525,14.485195,10.963663,31.707639,8.987720
6,0,0,0,0,0,0,0,0,0,0,...,39.312652,66.030987,38.170742,47.608252,22.613632,3.867694,13.218621,11.045743,33.894191,11.662195
7,0,0,0,0,0,0,0,0,0,0,...,33.574590,61.728455,32.830966,44.637842,19.941236,6.620987,10.905055,8.121156,33.800713,8.699502
8,0,0,0,0,0,0,0,0,0,0,...,41.410769,65.054452,37.168854,48.760881,23.455110,3.536101,11.234977,9.709289,32.370537,13.847700
9,0,0,0,0,0,0,0,0,1,0,...,33.858315,69.184135,36.981634,45.732295,21.930522,4.240701,9.916147,9.596940,35.853281,8.039272


In [4]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)

(2294, 116)
(2294,)
(405, 116)


Make objects of Dataset and DataLoader

In [5]:
class COVID19Dataset(Dataset):
    ''' Dataset for loading and preprocessing the COVID19 dataset '''
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        # Returns one sample at a time
        if self.y is None:
            # For training
            return self.x[idx]
        else:
            return self.x[idx], self.y[idx]

    def __len__(self):
        # Returns the size of the dataset
        return len(self.x)

In [6]:
train_dataset = COVID19Dataset(x_train, y_train)
test_dataset = COVID19Dataset(x_test)

- Group data into batches
- If you set shuffle=True, dataloader will permutes the indices of all samples automatically.
- We often set shuffle=True during training
- You can check the Advantage to shuffle a dataset (https://discuss.pytorch.org/t/advantage-to-shuffle-a-dataset/30584) if you are curious about why we should shuffle the data during training

In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

- All nn.Module objects are reusable as components of bigger networks ! That is how you build personnalized nets. The simplest way is to use the nn.Sequential class.
- The input dimension of our model will be 117
- The output of our model will be a scalar, which represents the predicting value of the percentage of new tested positive cases in the 5th day

In [8]:
# create a more customizable network module (equivalent here)
class My_model(torch.nn.Module):
    # you can use the layer sizes as initialization arguments if you want to
    def __init__(self, input_dim):
        super(My_model, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1) # (Batch, 1) --> (Batch)
        return x

model = My_model(input_dim = x_train.shape[1])

The network tracks parameters, and you can access them through the **parameters()** method, which returns a python generator.

In [9]:
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[ 0.0142,  0.0517,  0.0110,  ...,  0.0298,  0.0519, -0.0816],
        [ 0.0802,  0.0715, -0.0921,  ..., -0.0751, -0.0147, -0.0145],
        [-0.0265, -0.0159,  0.0625,  ...,  0.0401, -0.0282, -0.0870],
        ...,
        [-0.0881,  0.0158, -0.0002,  ...,  0.0434,  0.0416,  0.0284],
        [ 0.0696,  0.0266,  0.0414,  ..., -0.0313, -0.0163, -0.0379],
        [ 0.0397,  0.0172,  0.0073,  ...,  0.0126, -0.0662,  0.0729]],
       requires_grad=True)
Parameter containing:
tensor([-0.0594,  0.0776,  0.0758,  0.0567,  0.0667, -0.0266, -0.0835,  0.0618,
        -0.0836,  0.0350, -0.0165,  0.0724, -0.0544, -0.0697,  0.0767,  0.0054,
        -0.0763, -0.0564, -0.0890,  0.0595, -0.0565,  0.0504,  0.0505,  0.0381,
        -0.0165,  0.0732, -0.0417, -0.0557,  0.0883, -0.0116,  0.0687, -0.0054,
        -0.0036, -0.0496,  0.0506, -0.0390, -0.0322, -0.0177, -0.0007,  0.0037,
        -0.0361,  0.0578, -0.0427, -0.0395,  0.0866, -0.0064,  0.0103,  0.0733,
        -0.0667

We are doing a regression task, choosing mean square error as our loss function

In [10]:
criterion = nn.MSELoss(reduction='mean')

Optimizer
- We need to declare a optimizer that adjust network parameters in order to reduce error.
- Here we choose stochastic gradient descent as our optimization algorithm.

In [12]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)

Training Loop
- Get model prediction, compute gradient, update parameters and reset the gradient of model parameters.

In [13]:
for epoch in tqdm(range(500)):
    model.train() # set your model to train mode
    for x, y in train_loader:
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward() # compute the gredient through back-propagation
        optimizer.step() # update parameters
        optimizer.zero_grad() # set gradient to zero

100%|██████████| 500/500 [00:13<00:00, 36.72it/s]


Testing Loop

In [14]:
model.eval()
preds = []
for x in test_loader:
    with torch.no_grad():
        pred = model(x)
        preds.append(pred.numpy())
preds = np.concatenate(preds)

In [15]:
print("R-squared: "+str(explained_variance_score(y_test, preds)))
print("Mean Absolute Error: "+ str(mean_absolute_error(y_test, preds)))

R-squared: 0.9768602428941003
Mean Absolute Error: 0.7405982974163366


## Saving and Loading

In [16]:
# get dictionary of keys to weights using `state_dict`
print(model.state_dict().keys())

odict_keys(['layers.0.weight', 'layers.0.bias', 'layers.2.weight', 'layers.2.bias', 'layers.4.weight', 'layers.4.bias'])


In [ ]:
# save a dictionary
torch.save(model.state_dict(),'test.t7')
# load a dictionary
model.load_state_dict(torch.load('test.t7'))